# Model Evaluation with LMEval

Evaluate the quality impact of INT8 quantization by comparing original and compressed models.

### 1. Install and import libraries

**LMEval** is the industry standard framework for evaluating language models across standardized benchmarks.
It provides consistent, reproducible measurements of model quality.

In [ ]:
import os
from lm_eval import simple_evaluate

print("✓ Libraries ready")

### 2. Configure evaluation

In [ ]:
ORIGINAL_MODEL = "/shared-models/granite-4.0-350m"
COMPRESSED_MODEL = "/shared-models/granite-4.0-350m-int8-quantized/"
TASK = "arc_easy" # arc_easy: AI2 Reasoning Challenge (Easy) tests grade-school science reasoning  
LIMIT = 50  # Evaluating 50 samples provides quick feedback
BATCH_SIZE = "64" # Larger batches improve throughput

### 3. Evaluate original model

This establishes the baseline accuracy.
The original model uses BF16 precision (16-bit floating point), providing maximum quality at the cost of larger size and slower inference.

In [ ]:
results_orig = simple_evaluate(
    model="hf",
    model_args=f"pretrained={ORIGINAL_MODEL},dtype=auto",
    tasks=[TASK],
    device="cpu",
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

print("✓ Original model evaluation complete")
print(f"Accuracy: {results_orig['results'][TASK]['acc,none'] * 100:.2f}%")

### 4. Evaluate compressed model

The INT8 quantized model reduces weights from 16-bit to 8-bit. 
Manually load it with `transformers` because LMEval does not directly support the `compressed-tensors` format.

In [ ]:
# Load compressed model with transformers (supports compressed-tensors)
from transformers import AutoModelForCausalLM, AutoTokenizer
from lm_eval.models.huggingface import HFLM

compressed_path = os.path.abspath(COMPRESSED_MODEL)

# Load model and tokenizer
print("Loading compressed model...")
model = AutoModelForCausalLM.from_pretrained(
    compressed_path, 
    dtype="auto", 
    low_cpu_mem_usage=True
)
tokenizer = AutoTokenizer.from_pretrained(compressed_path)

# Wrap in lm-eval's HFLM wrapper
print("Preparing model for evaluation...")
lm = HFLM(
    pretrained=model,
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE
)

# Evaluate using the model object
print("Evaluating...")
results_comp = simple_evaluate(
    model=lm,
    tasks=[TASK],
    limit=LIMIT
)

print("✓ Compressed model evaluation complete")
print(f"Accuracy: {results_comp['results'][TASK]['acc,none'] * 100:.2f}%")